In [1]:
run_gridsearch = True
skip_best_model_validation = False
skip_best_model_test = False
verbose = False
GPU_SETTING = -1
NUM_ENSEMBLES = 1
NUM_ENSEMBLES_FINAL = 1  # ensemble count for final training (validation/test)
ADABOOST_ENSEMBLE = False # If ADABOOST_ENSEMBLE is True, it overrides BOOTSTRAP_MODEL
BOOTSTRAP_MODELS = False

BASIN = "warm_springs"
RES_1H = "mts_hourly"
RES_1D = "mts_daily"
RUN_LABEL = "CROSS_VAL_V3"
MODE = "mts"
READ_STAMP = "20250815T000000Z"

# --- Hyperparameter Selection Method ---
use_cv_for_selection = True  # False = single-split scoring, True = CV-averaged
CV_INTERVAL_LENGTH = 2
CV_VALIDATION_LENGTH = 1
CV_INTERVAL_MONTH = 'October'

GRID_RANK_METRICS = ["NSE_1D", "NSE_1H"]
GRID_RANK_WEIGHTS = [0.3, 0.7]

In [ ]:
hyperparam_space = {
    "hidden_size": [64, 128],
    "output_dropout": [0.1, 0.4],
    "seq_length_1D": [90],
    "seq_length_1H": [168, 336],
    "num_layers": [1],
    "epochs": [300],  # was [200] - plateau ES handles stopping
    "batch_size": [64],
    "schedule_pairs": [
        ((0.5, 0.25), (0.01, 0.005, 0.001))
    ]
}

In [3]:
import sys
import pandas as pd
import os
import itertools
from pathlib import Path
from tqdm import tqdm
import warnings
from datetime import datetime
warnings.simplefilter(action='ignore', category=FutureWarning)

In [4]:
current_dir = os.getcwd()
print(current_dir)

/Users/canruso/Desktop/UCB-USACE-RR-PROJECT/notebooks/basins/warm_springs


In [5]:
library_path = os.path.join('..', '..', '..','..','UCB-USACE-RR-PROJECT')
sys.path.insert(0, library_path)
print(sys.path)

['../../../../UCB-USACE-RR-PROJECT', '/Users/canruso/Desktop/UCB-USACE-RR-PROJECT/notebooks/basins/warm_springs', '/Users/canruso/Desktop', '/Users/canruso/miniforge3/envs/ESDL_LSTM/lib/python310.zip', '/Users/canruso/miniforge3/envs/ESDL_LSTM/lib/python3.10', '/Users/canruso/miniforge3/envs/ESDL_LSTM/lib/python3.10/lib-dynload', '', '/Users/canruso/miniforge3/envs/ESDL_LSTM/lib/python3.10/site-packages', '/Users/canruso/miniforge3/envs/ESDL_LSTM/lib/python3.10/site-packages/setuptools/_vendor']


In [6]:
from neuralhydrology.evaluation.metrics import *
from UCB_training.UCB_train import UCB_trainer
from UCB_training.UCB_utils import (fractional_multi_lr, write_paths, to_path_or_list, ensure_output_tree, set_active_context, data_dir, repo_root, get_output_dir, make_run_stamp, get_yaml_path, ctx_for, hparams_exists, save_hparams, load_hparams, runs_latest_path, archive_runs_json, read_csv_artifact, ensure_shared_tree, ensure_absolute_basin_files)
from UCB_training.UCB_plotting import (plot_timeseries_comparison, scatter_triptych_pngs_v3, ts_triptych_v3)

In [7]:
current_path = os.getcwd()
library_path = current_path.split('UCB-USACE-RR-PROJECT')[0] + 'UCB-USACE-RR-PROJECT'

In [8]:
RUNS_FILE = str(runs_latest_path(BASIN, MODE, RUN_LABEL))
SHOULD_STAMP = not (skip_best_model_validation and skip_best_model_test)
RUN_STAMP = make_run_stamp() if SHOULD_STAMP else None
ACTIVE_STAMP = RUN_STAMP if RUN_STAMP is not None else READ_STAMP

In [9]:
switch_ctx = ctx_for(BASIN, run_stamp=ACTIVE_STAMP, run_label=RUN_LABEL, append_stamp_to_filenames=False)
_SHARED = ensure_shared_tree(BASIN, MODE)
RUNS_PARENT = _SHARED / "runs" / (f"{RUN_LABEL}_{RUN_STAMP}" if RUN_STAMP else RUN_LABEL)

print("NH runs will be written under:")
print(RUNS_PARENT.resolve())

NH runs will be written under:
/Users/canruso/Desktop/UCB-USACE-RR-PROJECT/outputs/warm_springs/mts_shared/runs/CROSS_VAL_V3_20260221T060924Z


In [10]:
path_to_csv = data_dir()
path_to_yaml = get_yaml_path("warm_springs_mtslstm2")
path_to_physics_data_1D = path_to_csv / "WarmSprings_Inflow_daily_averaged.csv"
path_to_physics_data_1H = path_to_csv / "WarmSprings_Inflow_hourly.csv"

In [11]:
features_with_physics = [
    #from daily.csv
    "DRY CREEK 20 PRECIP-INC SCREENED",
    "DRY CREEK 20 ET-POTENTIAL RUN:BASIN AVERAGE 60 YR",
    "DRY CREEK 30 PRECIP-INC SCREENED",
    "DRY CREEK 30 ET-POTENTIAL RUN:BASIN AVERAGE 60 YR",
    "UKIAH CA HUMIDITY USAF-NOAA",
    "UKIAH CA SOLAR RADIATION USAF-NOAA",
    "UKIAH CA TEMPERATURE USAF-NOAA",
    "UKIAH CA WINDSPEED USAF-NOAA",
    "SANTA ROSA CA HUMIDITY USAF-NOAA",
    "SANTA ROSA CA SOLAR RADIATION USAF-NOAA",
    "SANTA ROSA CA TEMPERATURE USAF-NOAA",
    "SANTA ROSA CA WINDSPEED USAF-NOAA",
    #from Warm_Spring_Inflow.csv
    'Dry Creek 20 ET-POTENTIAL',
    'Dry Creek 20 FLOW',
    'Dry Creek 20 FLOW-BASE',
    'Dry Creek 20 INFILTRATION',
    'Dry Creek 20 PERC-SOIL',
    'Dry Creek 20 SATURATION FRACTION',
    'Dry Creek 30 ET-POTENTIAL',
    'Dry Creek 30 FLOW',
    'Dry Creek 30 FLOW-BASE',
    'Dry Creek 30 INFILTRATION',
    'Dry Creek 30 PERC-SOIL',
    'Dry Creek 30 SATURATION FRACTION',
    'Warm Springs Dam Inflow FLOW',
]

In [12]:
no_physics_results = []
physics_results = []

In [13]:
start_time = datetime.utcnow()
print("Start time:", start_time.strftime("%Y-%m-%d %H:%M:%S"))

Start time: 2026-02-21 06:09:24


In [14]:
import multiprocessing as mp
import itertools
import pandas as pd
from tqdm import tqdm
from UCB_training.grid_search_workers import run_single_experiment_nophysics, run_single_experiment_physics

print("run_gridsearch =", run_gridsearch)
print("hparams_exists =", hparams_exists(BASIN, MODE, RUN_LABEL))

hyperparam_names = []
for hp in hyperparam_space.keys():
    hyperparam_names.append(hp)

total_iters = 1
for name in hyperparam_names:
    total_iters *= len(hyperparam_space[name])

if run_gridsearch or not hparams_exists(BASIN, MODE, RUN_LABEL):

    all_combinations = list(itertools.product(*[hyperparam_space[hp] for hp in hyperparam_names]))

    num_cores = max(1, mp.cpu_count() - 1)
    print(f"\n[GRID SEARCH] Spawning {num_cores} workers\n")

    # NO PHYSICS GRID SEARCH

    task_args_no = [
        (idx, comb, hyperparam_names, path_to_csv, path_to_yaml,
         GPU_SETTING, RUNS_PARENT, RUN_LABEL, RUN_STAMP, verbose,
         UCB_trainer, fractional_multi_lr, NUM_ENSEMBLES, BOOTSTRAP_MODELS, ADABOOST_ENSEMBLE,
         True, True, use_cv_for_selection, CV_INTERVAL_MONTH, CV_INTERVAL_LENGTH, CV_VALIDATION_LENGTH)
        for idx, comb in enumerate(all_combinations)
    ]

    with mp.Pool(processes=num_cores) as pool:
        no_physics_results = list(tqdm(
            pool.imap(run_single_experiment_nophysics, task_args_no),
            total=len(all_combinations),
            desc="Grid No-Physics",
            unit="it",
            ncols=60,
            ascii=True
        ))

    df_no_physics = pd.DataFrame(no_physics_results)
    df_no_physics["_rank_score"] = sum(w * df_no_physics[m] for m, w in zip(GRID_RANK_METRICS, GRID_RANK_WEIGHTS))
    df_no_physics.sort_values(by="_rank_score", ascending=False, inplace=True)
    df_no_physics.reset_index(drop=True, inplace=True)

    print("\n✓ No-physics grid complete\n")

    # PHYSICS GRID SEARCH

    task_args_phys = [
        (idx, comb, hyperparam_names, path_to_csv, path_to_yaml,
         GPU_SETTING, RUNS_PARENT, RUN_LABEL, RUN_STAMP, verbose,
         UCB_trainer, fractional_multi_lr, NUM_ENSEMBLES, BOOTSTRAP_MODELS, ADABOOST_ENSEMBLE,
         features_with_physics, path_to_physics_data_1H,
         True, True, use_cv_for_selection, CV_INTERVAL_MONTH, CV_INTERVAL_LENGTH, CV_VALIDATION_LENGTH)
        for idx, comb in enumerate(all_combinations)
    ]

    with mp.Pool(processes=num_cores) as pool:
        physics_results = list(tqdm(
            pool.imap(run_single_experiment_physics, task_args_phys),
            total=len(all_combinations),
            desc="Grid Physics",
            unit="it",
            ncols=60,
            ascii=True
        ))

    df_physics = pd.DataFrame(physics_results)
    df_physics["_rank_score"] = sum(w * df_physics[m] for m, w in zip(GRID_RANK_METRICS, GRID_RANK_WEIGHTS))
    df_physics.sort_values(by="_rank_score", ascending=False, inplace=True)
    df_physics.reset_index(drop=True, inplace=True)

    print("\n✓ Physics grid complete\n")

    # save best params

    best_no_phys = df_no_physics.iloc[0].to_dict()
    best_phys = df_physics.iloc[0].to_dict()

    best_no_phys["model_type"] = "no_physics"
    best_phys["model_type"] = "physics"

    best_params_df = pd.DataFrame([best_no_phys, best_phys])

    save_hparams(
        best_df=best_params_df,
        basin=BASIN,
        mode=MODE,
        label=RUN_LABEL,
        run_stamp=RUN_STAMP,
        df_no=df_no_physics,
        df_phys=df_physics
    )

else:
    print("Skipping grid search!")

print("run_gridsearch =", run_gridsearch)
print("hparams_exists =", hparams_exists(BASIN, MODE, RUN_LABEL))

run_gridsearch = True
hparams_exists = False

[GRID SEARCH] Spawning 11 workers



Grid No-Physics:  12%|1| 1/8 [8:51:46<62:02:28, 31906.91s/itlibc++abi: terminating due to uncaught exception of type std::__1::system_error: condition_variable wait failed: Invalid argument
Grid No-Physics:  75%|7| 6/8 [13:48:30<3:09:28, 5684.48s/it]Process SpawnPoolWorker-12:
Grid No-Physics:  88%|8| 7/8 [59:55:26<8:33:38, 30818.01s/itProcess SpawnPoolWorker-9:
Process SpawnPoolWorker-11:
Process SpawnPoolWorker-10:
Traceback (most recent call last):
  File "/Users/canruso/miniforge3/envs/ESDL_LSTM/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/canruso/miniforge3/envs/ESDL_LSTM/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/canruso/miniforge3/envs/ESDL_LSTM/lib/python3.10/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Users/canruso/miniforge3/envs/ESDL_LSTM/lib/python3.10/multiprocessing/queues.py", line 364, in get
    with self._rlock

[GRID][PID 84455][003] NSE_1D=0.7632 NSE_1H=0.6663 | HP={'hidden_size': 64, 'output_dropout': 0.4, 'seq_length': {'1D': 90, '1H': 336}, 'num_layers': 1, 'epochs': 200, 'batch_size': 64, 'learning_rate': {0: 0.01, 100: 0.005, 150: 0.001}, 'save_weights_every': 200}
[GRID][PID 84458][006] NSE_1D=0.7411 NSE_1H=0.6746 | HP={'hidden_size': 128, 'output_dropout': 0.4, 'seq_length': {'1D': 90, '1H': 168}, 'num_layers': 1, 'epochs': 200, 'batch_size': 64, 'learning_rate': {0: 0.01, 100: 0.005, 150: 0.001}, 'save_weights_every': 200}
[GRID][PID 84453][001] NSE_1D=0.7368 NSE_1H=0.7144 | HP={'hidden_size': 64, 'output_dropout': 0.1, 'seq_length': {'1D': 90, '1H': 336}, 'num_layers': 1, 'epochs': 200, 'batch_size': 64, 'learning_rate': {0: 0.01, 100: 0.005, 150: 0.001}, 'save_weights_every': 200}
[GRID][PID 84452][000] NSE_1D=0.7648 NSE_1H=0.7162 | HP={'hidden_size': 64, 'output_dropout': 0.1, 'seq_length': {'1D': 90, '1H': 168}, 'num_layers': 1, 'epochs': 200, 'batch_size': 64, 'learning_rate': {

KeyboardInterrupt: 

In [ ]:
try:
    if run_gridsearch:
        print("\n[INFO] Using best_params_df from the just-completed grid search (ignoring READ_STAMP).")
    else:
        print("\nLoading best hyperparams from CSV...")
        best_params_df = load_hparams(BASIN, MODE, RUN_LABEL, stamp=READ_STAMP)
except FileNotFoundError as e:
    raise SystemExit(f"[ERROR] {e}  (Set run_gridsearch=True to generate it.)")

best_no_phys = best_params_df.query("model_type == 'no_physics'").iloc[0].to_dict()
best_phys = best_params_df.query("model_type == 'physics'").iloc[0].to_dict()

best_no_physics_params = {}
j = 0
while j < len(hyperparam_names):
    name = hyperparam_names[j]

    if name == "output_dropout":
        best_no_physics_params[name] = float(best_no_phys[name])
        j += 1

    elif name in ("seq_length_1D", "seq_length_1H"):

        best_no_physics_params["seq_length"] = {
            "1D": int(best_no_phys["seq_length_1D"]),
            "1H": int(best_no_phys["seq_length_1H"])}
        j += 2

    elif name == "schedule_pairs":
        j += 1

    else:
        best_no_physics_params[name] = int(best_no_phys[name])
        j += 1

if "learning_rate" in best_no_phys and pd.notna(best_no_phys["learning_rate"]):

    best_no_physics_params["learning_rate"] = eval(str(best_no_phys["learning_rate"]))

elif "schedule_pairs" in best_no_phys and pd.notna(best_no_phys["schedule_pairs"]):
    sp = best_no_phys["schedule_pairs"]
    if isinstance(sp, str):
        sp = eval(sp)
    fractions, rates = sp
    best_no_physics_params["learning_rate"] = fractional_multi_lr(
        epochs=int(best_no_physics_params["epochs"]),
        fractions=list(fractions),
        lrs=list(rates))

else:
    best_no_physics_params["learning_rate"] = {0: 0.01, 30: 0.005, 40: 0.001}

best_physics_params = {}
j = 0
while j < len(hyperparam_names):
    name = hyperparam_names[j]

    if name == "output_dropout":
        best_physics_params[name] = float(best_phys[name])
        j += 1

    elif name in ("seq_length_1D", "seq_length_1H"):
        best_physics_params["seq_length"] = {
            "1D": int(best_phys["seq_length_1D"]),
            "1H": int(best_phys["seq_length_1H"])}
        j += 2

    elif name == "schedule_pairs":
        j += 1

    else:
        best_physics_params[name] = int(best_phys[name])
        j += 1

if "learning_rate" in best_phys and pd.notna(best_phys["learning_rate"]):
    best_physics_params["learning_rate"] = eval(str(best_phys["learning_rate"]))

elif "schedule_pairs" in best_phys and pd.notna(best_phys["schedule_pairs"]):
    sp = best_phys["schedule_pairs"]
    if isinstance(sp, str):
        sp = eval(sp)
    fractions, rates = sp
    best_physics_params["learning_rate"] = fractional_multi_lr(
        epochs=int(best_physics_params["epochs"]),
        fractions=list(fractions),
        lrs=list(rates))

else:
    best_physics_params["learning_rate"] = {0: 0.01, 30: 0.005, 40: 0.001}

print("Loaded best hyperparams from CSV:")
print("Best NO-PHYS:", best_no_physics_params)
print("Best PHYS:", best_physics_params)

# #tempoverride:
# best_no_physics_params["epochs"] = 2
# best_physics_params["epochs"] = 2

In [ ]:
if not skip_best_model_validation:
    noPhysValTrainer = UCB_trainer(
        path_to_csv_folder=path_to_csv,
        yaml_path=path_to_yaml,
        hyperparams=best_no_physics_params,
        input_features=None,
        physics_informed=False,
        physics_data_file=None,
        hourly=True,
        extend_train_period=False,
        gpu=GPU_SETTING,
        is_mts = True,
        num_ensemble_members = NUM_ENSEMBLES_FINAL,
        adaboost_ensemble = ADABOOST_ENSEMBLE,
        bootstrap_model = BOOTSTRAP_MODELS,
        verbose=verbose,
        runs_parent=RUNS_PARENT,
        run_label=RUN_LABEL,
        run_stamp=RUN_STAMP,
        experiment_tag="no_phys_validation")
    
    noPhysValTrainer.train()
    noPhys_val_csv_1D, noPhys_val_metrics_1D = noPhysValTrainer.results(period="validation", mts_trk="1D")
    noPhys_val_csv_1H, noPhys_val_metrics_1H = noPhysValTrainer.results(period="validation", mts_trk="1H")
    print("NO-PHYS VAL 1D => NSE =", noPhys_val_metrics_1D.get("NSE", None))
    print("NO-PHYS VAL 1H => NSE =", noPhys_val_metrics_1H.get("NSE", None))

In [ ]:
if not skip_best_model_validation:
    physValTrainer = UCB_trainer(
        path_to_csv_folder=path_to_csv,
        yaml_path=path_to_yaml,
        hyperparams=best_physics_params,
        input_features=features_with_physics,
        physics_informed=True,
        physics_data_file=path_to_physics_data_1H,
        hourly=True,
        extend_train_period=False,
        gpu=GPU_SETTING,
        is_mts = True,
        num_ensemble_members = NUM_ENSEMBLES_FINAL,
        adaboost_ensemble = ADABOOST_ENSEMBLE,
        bootstrap_model = BOOTSTRAP_MODELS,
        verbose=verbose,
        runs_parent=RUNS_PARENT,
        run_label=RUN_LABEL,
        run_stamp=RUN_STAMP,
        experiment_tag="phys_validation")
    
    physValTrainer.train()
    phys_val_csv_1D, phys_val_metrics_1D = physValTrainer.results(period="validation", mts_trk="1D")
    phys_val_csv_1H, phys_val_metrics_1H = physValTrainer.results(period="validation", mts_trk="1H")
    print("PHYS VAL 1D => NSE =", phys_val_metrics_1D.get("NSE", None))
    print("PHYS VAL 1H => NSE =", phys_val_metrics_1H.get("NSE", None))

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_validation:
    plot_timeseries_comparison(source=(noPhys_val_csv_1D, phys_val_csv_1D, path_to_physics_data_1D), title="Warm Springs Basin MTS 1D Model Comparison (Validation)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_val_metrics_1D.csv", ts_out="warm_springs_mts_val_1D_combined_ts.csv", fig_out="warm_springs_mts_val_1D_model_comparison.png", legend_font=20, axis_font=22)
else:
    combined_df_val_1D = read_csv_artifact("warm_springs_mts_val_1D_combined_ts.csv", kind="csv", period="validation", stamp = READ_STAMP, run_label = RUN_LABEL)
    plot_timeseries_comparison(source=combined_df_val_1D, title="Warm Springs Basin MTS 1D Model Comparison (Validation)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_val_metrics_1D.csv", ts_out="warm_springs_mts_val_1D_combined_ts.csv", fig_out="warm_springs_mts_val_1D_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_validation:
    plot_timeseries_comparison(source=(noPhys_val_csv_1H, phys_val_csv_1H, path_to_physics_data_1H), title="Warm Springs Basin MTS 1H Model Comparison (Validation)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_val_metrics_1H.csv", ts_out="warm_springs_mts_val_1H_combined_ts.csv", fig_out="warm_springs_mts_val_1H_model_comparison.png", legend_font=20, axis_font=22)
else:
    combined_df_val_1H = read_csv_artifact("warm_springs_mts_val_1H_combined_ts.csv", kind="csv", period="validation", stamp = READ_STAMP, run_label = RUN_LABEL)
    plot_timeseries_comparison(source=combined_df_val_1H, title="Warm Springs Basin MTS 1H Model Comparison (Validation)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_val_metrics_1H.csv", ts_out="warm_springs_mts_val_1H_combined_ts.csv", fig_out="warm_springs_mts_val_1H_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
switch_ctx(RES_1D)

if skip_best_model_validation:
    val_metrics = read_csv_artifact("warm_springs_mts_val_metrics_1D.csv",  kind="metrics", period="validation", index_col=0, stamp = READ_STAMP, run_label = RUN_LABEL)
    print(val_metrics)

In [ ]:
switch_ctx(RES_1H)

if skip_best_model_validation:
    val_metrics = read_csv_artifact("warm_springs_mts_val_metrics_1H.csv",  kind="metrics", period="validation", index_col=0, stamp = READ_STAMP, run_label = RUN_LABEL)
    print(val_metrics)

# Test Period

In [ ]:
if not skip_best_model_test:
    print("\nTraining No-Physics MTS model for test period...")
    mtsNoPhysicsTest = UCB_trainer(
        path_to_csv_folder=path_to_csv,
        yaml_path=path_to_yaml,
        hyperparams=best_no_physics_params,
        input_features=None,
        physics_informed=False,
        physics_data_file=None,
        hourly=True,
        extend_train_period=True,
        gpu=GPU_SETTING,
        is_mts = True,
        verbose=verbose,
        num_ensemble_members = NUM_ENSEMBLES_FINAL,
        adaboost_ensemble = ADABOOST_ENSEMBLE,
        bootstrap_model = BOOTSTRAP_MODELS,
        runs_parent=RUNS_PARENT,
        run_label=RUN_LABEL,
        run_stamp=RUN_STAMP,
        experiment_tag="no_phys_test")

    ensure_absolute_basin_files(mtsNoPhysicsTest, BASIN)
    
    mtsNoPhysicsTest.train()
    no_physics_test_csv_1D, no_physics_test_metrics_1D = mtsNoPhysicsTest.results(period="test", mts_trk="1D")
    no_physics_test_csv_1H, no_physics_test_metrics_1H = mtsNoPhysicsTest.results(period="test", mts_trk="1H")
    print("\n[No-Physics Test] NSE @1D =", no_physics_test_metrics_1D.get("NSE", None))
    print("[No-Physics Test] NSE @1H =", no_physics_test_metrics_1H.get("NSE", None))

In [ ]:
if not skip_best_model_test:
    print("\nTraining Physics MTS model for test period...")
    mtsPhysicsTest = UCB_trainer(
        path_to_csv_folder=path_to_csv,
        yaml_path=path_to_yaml,
        hyperparams=best_physics_params,
        input_features=features_with_physics,
        physics_informed=True,
        physics_data_file=path_to_physics_data_1H,
        hourly=True,
        extend_train_period=True,
        gpu=GPU_SETTING,
        is_mts = True,
        num_ensemble_members = NUM_ENSEMBLES_FINAL,
        adaboost_ensemble = ADABOOST_ENSEMBLE,
        bootstrap_model = BOOTSTRAP_MODELS,
        verbose=verbose,
        runs_parent=RUNS_PARENT,
        run_label=RUN_LABEL,
        run_stamp=RUN_STAMP,
        experiment_tag="phys_test")

    ensure_absolute_basin_files(mtsPhysicsTest, BASIN)
    
    mtsPhysicsTest.train()
    physics_test_csv_1D, physics_test_metrics_1D = mtsPhysicsTest.results(period="test", mts_trk="1D")
    physics_test_csv_1H, physics_test_metrics_1H = mtsPhysicsTest.results(period="test", mts_trk="1H")
    print("\n[Physics Test] NSE @1D =", physics_test_metrics_1D.get("NSE", None))
    print("[Physics Test] NSE @1H =", physics_test_metrics_1H.get("NSE", None))

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1D, physics_test_csv_1D, path_to_physics_data_1D), title="Warm Springs Basin MTS 1D Model Comparison (Test)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_test_metrics_1D.csv", ts_out="warm_springs_mts_test_1D_combined_ts.csv", fig_out="warm_springs_mts_test_1D_model_comparison.png", legend_font=20, axis_font=22)
else:
    combined_df_test_1D = read_csv_artifact("warm_springs_mts_test_1D_combined_ts.csv", kind="csv", period="test", stamp = READ_STAMP, run_label = RUN_LABEL)
    plot_timeseries_comparison(source=combined_df_test_1D, title="Warm Springs Basin MTS 1D Model Comparison (Test)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_test_metrics_1D.csv", ts_out="warm_springs_mts_test_1D_combined_ts.csv", fig_out="warm_springs_mts_test_1D_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1D, physics_test_csv_1D, path_to_physics_data_1D), title="Warm Springs Basin MTS 1D Model Comparison (Test)", backend="plotly", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_test_metrics_1D.csv", ts_out="warm_springs_mts_test_1D_combined_ts.csv", fig_out="warm_springs_mts_test_1D_model_comparison.png", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1D, title="Warm Springs Basin MTS 1D Model Comparison (Test)", backend="plotly", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_test_metrics_1D.csv", ts_out="warm_springs_mts_test_1D_combined_ts.csv", fig_out="warm_springs_mts_test_1D_model_comparison.png", legend_font=12, axis_font=22)

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1H, physics_test_csv_1H, path_to_physics_data_1H), title="Warm Springs Basin MTS 1H Model Comparison (Test)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_test_metrics_1H.csv", ts_out="warm_springs_mts_test_1H_combined_ts.csv", fig_out="warm_springs_mts_test_1H_model_comparison.png", legend_font=20, axis_font=22)
else:
    combined_df_test_1H = read_csv_artifact("warm_springs_mts_test_1H_combined_ts.csv", kind="csv", period="test", stamp = READ_STAMP, run_label = RUN_LABEL)
    plot_timeseries_comparison(source=combined_df_test_1H, title="Warm Springs Basin MTS 1H Model Comparison (Test)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_test_metrics_1H.csv", ts_out="warm_springs_mts_test_1H_combined_ts.csv", fig_out="warm_springs_mts_test_1H_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1H, physics_test_csv_1H, path_to_physics_data_1H), title="Warm Springs Basin MTS 1H Model Comparison (Test)", backend="plotly", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_test_metrics_1H.csv", ts_out="warm_springs_mts_test_1H_combined_ts.csv", fig_out="warm_springs_mts_test_1H_model_comparison.png", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1H, title="Warm Springs Basin MTS 1H Model Comparison (Test)", backend="plotly", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_mts_test_metrics_1H.csv", ts_out="warm_springs_mts_test_1H_combined_ts.csv", fig_out="warm_springs_mts_test_1H_model_comparison.png", legend_font=12, axis_font=22)

In [ ]:
switch_ctx(RES_1D)

test_metrics = read_csv_artifact("warm_springs_mts_test_metrics_1D.csv", kind="metrics", period="test", index_col=0, stamp = READ_STAMP, run_label = RUN_LABEL)
print(test_metrics)

In [ ]:
switch_ctx(RES_1H)

test_metrics = read_csv_artifact("warm_springs_mts_test_metrics_1H.csv", kind="metrics", period="test", index_col=0, stamp = READ_STAMP, run_label = RUN_LABEL)
print(test_metrics)

In [ ]:
if not skip_best_model_validation:
    write_paths("no_physics_val", noPhysValTrainer, filename = RUNS_FILE)
    write_paths("physics_val", physValTrainer, filename = RUNS_FILE)

if not skip_best_model_test:
    write_paths("no_physics", mtsNoPhysicsTest, filename = RUNS_FILE)
    write_paths("physics", mtsPhysicsTest, filename = RUNS_FILE)
    archived_path = archive_runs_json(Path(RUNS_FILE), BASIN, MODE, RUN_LABEL, RUN_STAMP)

In [ ]:
end_time = datetime.utcnow()
print("End time:", end_time.strftime("%Y-%m-%d %H:%M:%S"))
print("Total time:", end_time - start_time)

##### Additional Plots

In [ ]:
if skip_best_model_validation:
    switch_ctx(RES_1D)
    combined_df_daily_val = read_csv_artifact("warm_springs_mts_val_1D_combined_ts.csv", kind="csv", period="validation", stamp = READ_STAMP, run_label = RUN_LABEL)
    switch_ctx(RES_1H)
    combined_df_hourly_val = read_csv_artifact("warm_springs_mts_val_1H_combined_ts.csv", kind="csv", period="validation", stamp = READ_STAMP, run_label = RUN_LABEL)
    
if skip_best_model_test:
    switch_ctx(RES_1D)
    combined_df_daily = read_csv_artifact("warm_springs_mts_test_1D_combined_ts.csv", kind="csv", period="test", stamp = READ_STAMP, run_label = RUN_LABEL)
    switch_ctx(RES_1H)
    combined_df_hourly = read_csv_artifact("warm_springs_mts_test_1H_combined_ts.csv", kind="csv", period="test", stamp = READ_STAMP, run_label = RUN_LABEL)

In [ ]:
metric_list = ["NSE", "PBIAS"]

wettest_start_val = "2003-10-01"
wettest_end_val = "2004-09-30"
dryest_start_val = "2004-10-01"
dryest_end_val = "2005-09-30"
wettest_start_test = "2005-10-01"
wettest_end_test = "2006-09-30"
dryest_start_test = "2008-10-01"
dryest_end_test = "2009-09-30"

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_validation:
    plot_timeseries_comparison(source=(noPhys_val_csv_1D, phys_val_csv_1D, path_to_physics_data_1D), title="Warm Springs Daily MTS Validation Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_mts_val_metrics_1D.csv", ts_out="warm_springs_mts_val_1D_combined_ts.csv", fig_out="warm_springs_mts_val_plot_1D.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_daily_val, title="Warm Springs Daily MTS Validation Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_mts_val_metrics_1D.csv", ts_out="warm_springs_mts_val_1D_combined_ts.csv", fig_out="warm_springs_mts_val_plot_1D.png", legend_font=20, axis_font=22)

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_validation:
    plot_timeseries_comparison(source=(noPhys_val_csv_1H, phys_val_csv_1H, path_to_physics_data_1H), title="Warm Springs Hourly MTS Validation Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_mts_val_metrics_1H.csv", ts_out="warm_springs_mts_val_1H_combined_ts.csv", fig_out="warm_springs_mts_val_1H_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_hourly_val, title="Warm Springs Hourly MTS Validation Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_mts_val_metrics_1H.csv", ts_out="warm_springs_mts_val_1H_combined_ts.csv", fig_out="warm_springs_mts_val_1H_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1D, physics_test_csv_1D, path_to_physics_data_1D), title="Warm Springs Daily MTS Test Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_mts_test_metrics_1D.csv", ts_out="warm_springs_mts_test_1D_combined_ts.csv", fig_out="warm_springs_mts_test_1D_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1D, title="Warm Springs Daily MTS Test Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_mts_test_metrics_1D.csv", ts_out="warm_springs_mts_test_1D_combined_ts.csv", fig_out="warm_springs_mts_test_1D_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1D, physics_test_csv_1D, path_to_physics_data_1D), title="Warm Springs Daily MTS Test Timeseries - Interactive", backend="plotly", metrics=metric_list, metrics_out="warm_springs_mts_test_metrics_1D.csv", ts_out="warm_springs_mts_test_1D_combined_ts.csv", fig_out="warm_springs_mts_test_1D_model_comparison.png", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1D, title="Warm Springs Daily MTS Test Timeseries - Interactive", backend="plotly", metrics=metric_list, metrics_out="warm_springs_mts_test_metrics_1D.csv", ts_out="warm_springs_mts_test_1D_combined_ts.csv", fig_out="warm_springs_mts_test_1D_model_comparison.png", legend_font=12, axis_font=22)

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1H, physics_test_csv_1H, path_to_physics_data_1H), title="Warm Springs Hourly MTS Test Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_mts_test_metrics_1H.csv", ts_out="warm_springs_mts_test_1H_combined_ts.csv", fig_out="warm_springs_mts_test_1H_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1H, title="Warm Springs Hourly MTS Test Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_mts_test_metrics_1H.csv", ts_out="warm_springs_mts_test_1H_combined_ts.csv", fig_out="warm_springs_mts_test_1H_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1H, physics_test_csv_1H, path_to_physics_data_1H), title="Warm Springs Hourly MTS Test Timeseries - Interactive", backend="plotly", metrics=metric_list, metrics_out="warm_springs_mts_test_metrics_1H.csv", ts_out="warm_springs_mts_test_1H_combined_ts.csv", fig_out="warm_springs_mts_test_1H_model_comparison.png", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1H, title="Warm Springs Hourly MTS Test Timeseries - Interactive", backend="plotly", metrics=metric_list, metrics_out="warm_springs_mts_test_metrics_1H.csv", ts_out="warm_springs_mts_test_1H_combined_ts.csv", fig_out="warm_springs_mts_test_1H_model_comparison.png", legend_font=12, axis_font=22)

##### Ensemble Performance Overview

In [ ]:
if NUM_ENSEMBLES_FINAL > 1:
    import numpy as np

    def load_hms(path, basin_name="calpella"):
        path = Path(path)
        print("\nLoading HMS from:")
        print(path)

        raw = pd.read_csv(path, header=None, dtype=str)

        hdr = raw.apply(lambda r: r.str.contains("date", case=False, na=False).any(), axis=1).idxmax()

        df = raw.iloc[hdr:].copy()
        df.columns = df.iloc[0].str.strip().str.lower()
        df = df.iloc[1:].copy()

        df.columns = [c.strip().lower() for c in df.columns]

        date_col = next((c for c in df.columns if "date" in c), None)
        time_col = next((c for c in df.columns if "time" in c), None)

        if date_col is None:
            raise RuntimeError("No Date column found in HMS CSV")

        if time_col is not None and df[time_col].notna().any():

            df[time_col] = df[time_col].fillna("00:00:00").str.strip()

            mask24 = df[time_col] == "24:00:00"
            if mask24.any():
                df.loc[mask24, date_col] = (
                    pd.to_datetime(df.loc[mask24, date_col], format="%d-%b-%y")
                    + pd.Timedelta(days=1)
                ).dt.strftime("%d-%b-%y")
                df.loc[mask24, time_col] = "00:00:00"

            df["date"] = (
                pd.to_datetime(df[date_col], format="%d-%b-%y", errors="coerce") +
                pd.to_timedelta(df[time_col])
            )

        else:
            df["date"] = pd.to_datetime(df[date_col], format="%d-%b-%y", errors="coerce")

        df.dropna(subset=["date"], inplace=True)

        basin_cols = [c for c in df.columns if "basin" in c]
        if basin_cols:
            bc = basin_cols[0]
            print(f"Filtering by basin column: {bc}")
            df = df[df[bc].astype(str).str.lower().str.contains(basin_name)]

        df = df.sort_values("date").reset_index(drop=True)

        print("Rows:", len(df))
        print("Columns:", list(df.columns))

        return df

    def load_hms_simple(path):
        df = load_hms(path)

        flow_col = next(c for c in df.columns if "gage flow" in c.lower())

        df = df[["date", flow_col]].rename(columns={
            "date": "Date",
            flow_col: "HMS"
        })

        df["Date"] = pd.to_datetime(df["Date"]).dt.floor("H")

        return df


    def nse(obs, sim):
        obs = np.asarray(obs, dtype=float)
        sim = np.asarray(sim, dtype=float)

        mask = np.isfinite(obs) & np.isfinite(sim)
        obs = obs[mask]
        sim = sim[mask]

        return 1 - np.sum((sim - obs)**2) / np.sum((obs - obs.mean())**2)


    def load_ensemble_mean(folder, split, freq):

        dfs = []

        for m in sorted(folder.glob("member_*")):
            p = m / f"{split.lower()}_{freq}.csv"
            if not p.exists():
                continue

            df = pd.read_csv(p, parse_dates=["Date"])

            df = df[["Date", "Predicted", "Observed"]]

            df["Predicted"] = pd.to_numeric(df["Predicted"], errors="coerce")
            df["Observed"] = pd.to_numeric(df["Observed"], errors="coerce")

            df = df.rename(columns={"Predicted": m.name})

            dfs.append(df)

        if not dfs:
            raise RuntimeError(f"No ensemble CSVs in {folder}")

        merged = dfs[0]

        for df in dfs[1:]:
            merged = merged.merge(df, on=["Date", "Observed"], how="inner")

        member_cols = [c for c in merged.columns if c not in ["Date", "Observed"]]

        merged["ensemble"] = merged[member_cols].mean(axis=1)

        merged = merged.dropna(subset=["Observed", "ensemble"])

        return merged[["Date", "Observed", "ensemble"]]


    def eval_split(split, freq, folder, hms_df):

        df = load_ensemble_mean(folder, split, freq)

        df = df.merge(hms_df, on="Date", how="inner")

        df["HMS"] = pd.to_numeric(df["HMS"], errors="coerce")

        df = df.dropna(subset=["Observed", "ensemble", "HMS"])

        print(f"{split}-{freq}: rows used = {len(df)}")

        return {
            "HMS": nse(df["Observed"].values, df["HMS"].values),
            "MODEL": nse(df["Observed"].values, df["ensemble"].values)
        }

    def eval_single_member(no_phys_folder, phys_folder, split, freq, hms_df, member_name):

        def load_member(folder):
            p = folder / member_name / f"{split.lower()}_{freq}.csv"
            if not p.exists():
                raise RuntimeError(f"Missing {p}")

            df = pd.read_csv(p, parse_dates=["Date"])
            df = df[["Date", "Predicted", "Observed"]]

            df["Predicted"] = pd.to_numeric(df["Predicted"], errors="coerce")
            df["Observed"] = pd.to_numeric(df["Observed"], errors="coerce")

            return df

        # LSTM (no physics)
        df_lstm = load_member(no_phys_folder)

        # PI LSTM
        df_pi = load_member(phys_folder).rename(columns={"Predicted": "PI"})

        df = df_lstm.merge(df_pi[["Date", "PI"]], on="Date", how="inner")
        df = df.merge(hms_df, on="Date", how="inner")

        df["HMS"] = pd.to_numeric(df["HMS"], errors="coerce")

        df = df.dropna(subset=["Observed", "Predicted", "PI", "HMS"])

        return {
            "HMS": nse(df["Observed"], df["HMS"]),
            "LSTM": nse(df["Observed"], df["Predicted"]),
            "PI": nse(df["Observed"], df["PI"])
        }

    from sklearn.linear_model import Ridge
    import numpy as np

    def fit_ridge_weights(val_pred_matrix, y_val, alpha=1.0, nonneg=False):
        """
        val_pred_matrix: shape (T, M)
        y_val: shape (T,)
        Returns weights shape (M,)
        """
        model = Ridge(alpha=alpha, fit_intercept=False)
        model.fit(val_pred_matrix, y_val)
        w = model.coef_.copy()

        if nonneg:
            w = np.maximum(w, 0)

        # normalize to sum to 1 for stability
        if w.sum() != 0:
            w = w / w.sum()

        return w

    def load_member_matrix(folder, split, freq):
        dfs = []
        names = []
        for m in sorted(folder.glob("member_*")):
            p = m / f"{split.lower()}_{freq}.csv"
            if not p.exists():
                continue
            df = pd.read_csv(p, parse_dates=["Date"])[["Date", "Predicted", "Observed"]]
            df["Predicted"] = pd.to_numeric(df["Predicted"], errors="coerce")
            df["Observed"] = pd.to_numeric(df["Observed"], errors="coerce")
            df = df.dropna(subset=["Predicted", "Observed"])
            df = df.rename(columns={"Predicted": m.name})
            dfs.append(df)
            names.append(m.name)

        merged = dfs[0]
        for df in dfs[1:]:
            merged = merged.merge(df, on=["Date", "Observed"], how="inner")

        X = merged[names].values  # (T, M)
        y = merged["Observed"].values
        dates = merged["Date"].values
        return dates, y, X, names

    def eval_ridge_weighted(val_folder, test_folder, freq, hms_df, alpha=1.0):
        # fit on validation
        _, y_val, X_val, names = load_member_matrix(val_folder, "Validation", freq)
        w = fit_ridge_weights(X_val, y_val, alpha=alpha, nonneg=True)

        # apply to test
        dates_t, y_t, X_t, _ = load_member_matrix(test_folder, "Test", freq)
        y_hat = X_t @ w

        df = pd.DataFrame({"Date": pd.to_datetime(dates_t), "Observed": y_t, "ensemble": y_hat})
        df = df.merge(hms_df, on="Date", how="inner").dropna()

        return w, {
            "HMS": nse(df["Observed"], df["HMS"]),
            "MODEL": nse(df["Observed"], df["ensemble"])
        }, names

    ROOT = Path(RUNS_PARENT, "ensemble_predictions/BASELINE")

    folders = {
        "phys_validation": ROOT / "phys_validation",
        "no_phys_validation": ROOT / "no_phys_validation",
        "phys_test": ROOT / "phys_test",
        "no_phys_test": ROOT / "no_phys_test",
    }

    HMS_1D = path_to_physics_data_1D
    HMS_1H = path_to_physics_data_1H
    hms1d = load_hms_simple(HMS_1D)
    hms1h = load_hms_simple(HMS_1H)

    members = ["member_01", "member_02",  "member_03",  "member_04",  "member_05"]

    member_tables = {m: [] for m in members}
    ensemble_results = {}

    for split in ["Validation", "Test"]:
        for freq, hms_df in [("1D", hms1d), ("1H", hms1h)]:

            no_phys = folders[f"no_phys_{split.lower()}"]
            phys = folders[f"phys_{split.lower()}"]

            for m in members:
                r = eval_single_member(no_phys, phys, split, freq, hms_df, m)

                member_tables[m].append([
                    f"{split} - {freq}",
                    round(r["HMS"], 3),
                    round(r["LSTM"], 3),
                    round(r["PI"], 3)
                ])

            lstm = eval_split(split, freq, no_phys, hms_df)
            pi   = eval_split(split, freq, phys, hms_df)

            ensemble_results[f"{split}-{freq}"] = {
                "HMS": lstm["HMS"],
                "LSTM": lstm["MODEL"],
                "PI": pi["MODEL"]
            }

    # ================= MEMBER TABLES =================

    for m in members:
        print(f"\n========= NSE TABLE FOR {m.upper()} =========\n")

        df_m = pd.DataFrame(
            member_tables[m],
            columns=["Run Type", "HMS Prediction", "LSTM Prediction", "PI LSTM Prediction"]
        )

        display(df_m)

    # ================= COMBINED TABLE =================

    rows = []

    for k in ["Validation-1D","Validation-1H","Test-1D","Test-1H"]:
        r = ensemble_results[k]
        rows.append([
            k.replace("-", " - "),
            round(r["HMS"],3),
            round(r["LSTM"],3),
            round(r["PI"],3)
        ])

    df_out = pd.DataFrame(
        rows,
        columns=["Run Type","HMS Prediction","LSTM Prediction","PI LSTM Prediction"]
    )

    print("\n========= COMBINED ENSEMBLE NSE TABLE =========\n")
    display(df_out)

    # ================= VALIDATION WEIGHTS =================

    rows = []
    for freq, hms_df in [("1D", hms1d), ("1H", hms1h)]:
        # LSTM (no-phys)
        w_lstm, res_lstm, names_lstm = eval_ridge_weighted(
            folders["no_phys_validation"], folders["no_phys_test"], freq, hms_df, alpha=10.0
        )

        # PI (phys)
        w_pi, res_pi, names_pi = eval_ridge_weighted(
            folders["phys_validation"], folders["phys_test"], freq, hms_df, alpha=10.0
        )

        rows.append([
            f"Test - {freq}",
            round(res_lstm["HMS"], 3),
            round(res_lstm["MODEL"], 3),
            round(res_pi["MODEL"], 3),
            {n: round(float(w), 3) for n, w in zip(names_lstm, w_lstm)},
            {n: round(float(w), 3) for n, w in zip(names_pi, w_pi)},
        ])

    df_ridge = pd.DataFrame(rows, columns=[
        "Run Type", "HMS Prediction", "Ridge-Weighted LSTM", "Ridge-Weighted PI LSTM",
        "LSTM Weights", "PI Weights"
    ])

    print("\n========= RIDGE-LEARNED WEIGHTS (VAL → TEST) =========\n")
    display(df_ridge)

In [ ]:
# EVERYTHING BELOW HERE MEASURES EXTREME YEARS.

raise SystemExit("EVERYTHING BELOW HERE MEASURES EXTREME YEARS WITH HARDCODED VALUES AND DOESNT MAKE SENSE FOR SYNTHETIC")

##### Wettest Year Performance

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_validation:
    plot_timeseries_comparison(source=(noPhys_val_csv_1D, phys_val_csv_1D, path_to_physics_data_1D), title="Warm Springs Daily MTS Wettest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_val, end_date=wettest_end_val, metrics_out="warm_springs_mts_wet_val_metrics_1D.csv", ts_out="warm_springs_mts_wet_val_1D_combined_ts.csv", fig_out="warm_springs_mts_wet_val_1D_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_daily_val, title="Warm Springs Daily MTS Wettest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_val, end_date=wettest_end_val, metrics_out="warm_springs_mts_wet_val_metrics_1D.csv", ts_out="warm_springs_mts_wet_val_1D_combined_ts.csv", fig_out="warm_springs_mts_wet_val_1D_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_validation:
    plot_timeseries_comparison(source=(noPhys_val_csv_1H, phys_val_csv_1H, path_to_physics_data_1H), title="Warm Springs Hourly MTS Wettest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_val, end_date=wettest_end_val, metrics_out="warm_springs_mts_wet_val_metrics_1H.csv", ts_out="warm_springs_mts_wet_val_1H_combined_ts.csv", fig_out="warm_springs_mts_wet_val_1H_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_hourly_val, title="Warm Springs Hourly MTS Wettest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_val, end_date=wettest_end_val, metrics_out="warm_springs_mts_wet_val_metrics_1H.csv", ts_out="warm_springs_mts_wet_val_1H_combined_ts.csv", fig_out="warm_springs_mts_wet_val_1H_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1D, physics_test_csv_1D, path_to_physics_data_1D), title="Warm Springs Daily MTS Wettest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_mts_wet_test_metrics_1D.csv", ts_out="warm_springs_mts_wet_test_1D_combined_ts.csv", fig_out="warm_springs_mts_wet_test_1D_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1D, title="Warm Springs Daily MTS Wettest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_mts_wet_test_metrics_1D.csv", ts_out="warm_springs_mts_wet_test_1D_combined_ts.csv", fig_out="warm_springs_mts_wet_test_1D_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1D, physics_test_csv_1D, path_to_physics_data_1D), title="Warm Springs Daily MTS Wettest Year Test Timeseries - Interactive", backend="plotly", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_mts_wet_test_metrics_1D.csv", ts_out="warm_springs_mts_wet_test_1D_combined_ts.csv", fig_out="warm_springs_mts_wet_test_1D_model_comparison.png", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1D, title="Warm Springs Daily MTS Wettest Year Test Timeseries - Interactive", backend="plotly", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_mts_wet_test_metrics_1D.csv", ts_out="warm_springs_mts_wet_test_1D_combined_ts.csv", fig_out="warm_springs_mts_wet_test_1D_model_comparison.png", legend_font=12, axis_font=22)

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1H, physics_test_csv_1H, path_to_physics_data_1H), title="Warm Springs Hourly MTS Wettest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_mts_wet_test_metrics_1H.csv", ts_out="warm_springs_mts_wet_test_1H_combined_ts.csv", fig_out="warm_springs_mts_wet_test_1H_model_comparison.png", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1H, title="warm_springs Hourly MTS Wettest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_mts_wet_test_metrics_1H.csv", ts_out="warm_springs_mts_wet_test_1H_combined_ts.csv", fig_out="warm_springs_mts_wet_test_1H_model_comparison.png", legend_font=12, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1H, physics_test_csv_1H, path_to_physics_data_1H), title="Warm Springs Hourly MTS Wettest Year Test Timeseries - Interactive", backend="plotly", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_mts_wet_test_metrics_1H.csv", ts_out="warm_springs_mts_wet_test_1H_combined_ts.csv", fig_out="warm_springs_mts_wet_test_1H_model_comparison.png", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1H, title="Warm Springs Hourly MTS Wettest Year Test Timeseries - Interactive", backend="plotly", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_mts_wet_test_metrics_1H.csv", ts_out="warm_springs_mts_wet_test_1H_combined_ts.csv", fig_out="warm_springs_mts_wet_test_1H_model_comparison.png", legend_font=12, axis_font=22)

##### Driest Year Performance

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_validation:
    plot_timeseries_comparison(source=(noPhys_val_csv_1D, phys_val_csv_1D, path_to_physics_data_1D), title="Warm Springs Daily MTS Dryest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_val, end_date=dryest_end_val, metrics_out="warm_springs_mts_dry_val_metrics_1D.csv", ts_out="warm_springs_mts_dry_val_1D_combined_ts.csv", fig_out="warm_springs_mts_dry_val_1D_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_daily_val, title="Warm Springs Daily MTS Dryest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_val, end_date=dryest_end_val, metrics_out="warm_springs_mts_dry_val_metrics_1D.csv", ts_out="warm_springs_mts_dry_val_1D_combined_ts.csv", fig_out="warm_springs_mts_dry_val_1D_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_validation:
    plot_timeseries_comparison(source=(noPhys_val_csv_1H, phys_val_csv_1H, path_to_physics_data_1H), title="Warm Springs Hourly MTS Dryest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_val, end_date=dryest_end_val, metrics_out="warm_springs_mts_dry_val_metrics_1H.csv", ts_out="warm_springs_mts_dry_val_1H_combined_ts.csv", fig_out="warm_springs_mts_dry_val_1H_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_hourly_val, title="Warm Springs Hourly MTS Dryest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_val, end_date=dryest_end_val, metrics_out="warm_springs_mts_dry_val_metrics_1H.csv", ts_out="warm_springs_mts_dry_val_1H_combined_ts.csv", fig_out="warm_springs_mts_dry_val_1H_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1D, physics_test_csv_1D, path_to_physics_data_1D), title="Warm Springs Daily MTS Dryest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_mts_dry_test_metrics_1D.csv", ts_out="warm_springs_mts_dry_test_1D_combined_ts.csv", fig_out="warm_springs_mts_dry_test_1D_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1D, title="Warm Springs Daily MTS Dryest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_mts_dry_test_metrics_1D.csv", ts_out="warm_springs_mts_dry_test_1D_combined_ts.csv", fig_out="warm_springs_mts_dry_test_1D_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1D, physics_test_csv_1D, path_to_physics_data_1D), title="Warm Springs Daily MTS Dryest Year Test Timeseries - Interactive", backend="plotly", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_mts_dry_test_metrics_1D.csv", ts_out="warm_springs_mts_dry_test_1D_combined_ts.csv", fig_out="warm_springs_mts_dry_test_1D_model_comparison.png", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1D, title="Warm Springs Daily MTS Dryest Year Test Timeseries - Interactive", backend="plotly", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_mts_dry_test_metrics_1D.csv", ts_out="warm_springs_mts_dry_test_1D_combined_ts.csv", fig_out="warm_springs_mts_dry_test_1D_model_comparison.png", legend_font=12, axis_font=22)

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1H, physics_test_csv_1H, path_to_physics_data_1H), title="Warm Springs Hourly MTS Dryest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_mts_dry_test_metrics_1H.csv", ts_out="warm_springs_mts_dry_test_1H_combined_ts.csv", fig_out="warm_springs_mts_dry_test_1H_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1H, title="Warm Springs Hourly MTS Dryest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_mts_dry_test_metrics_1H.csv", ts_out="warm_springs_mts_dry_test_1H_combined_ts.csv", fig_out="warm_springs_mts_dry_test_1H_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv_1H, physics_test_csv_1H, path_to_physics_data_1H), title="Warm Springs Hourly MTS Dryest Year Test Timeseries - Interactive", backend="plotly", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_mts_dry_test_metrics_1H.csv", ts_out="warm_springs_mts_dry_test_1H_combined_ts.csv", fig_out="warm_springs_mts_dry_test_1H_model_comparison.png", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_test_1H, title="Warm Springs Hourly MTS Dryest Year Test Timeseries - Interactive", backend="plotly", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_mts_dry_test_metrics_1H.csv", ts_out="warm_springs_mts_dry_test_1H_combined_ts.csv", fig_out="warm_springs_mts_dry_test_1H_model_comparison.png", legend_font=12, axis_font=22)

##### Gridded Timeseries Plots - Validation and Test

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_validation:
    ts_triptych_v3((noPhys_val_csv_1D, phys_val_csv_1D, path_to_physics_data_1D),wet_start=wettest_start_val, wet_end=wettest_end_val, dry_start=dryest_start_val, dry_end=dryest_end_val, save_path="warm_springs_mts_daily_TS_validation.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d-%b-%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title="Warm Springs MTS Daily Validation Period Timeseries Across Models", main_title_font=14, main_title_y=0.99, main_title_pad=0.05, row_titles=("Full Validation period","Most-wet water-year","Most-dry water-year"))

else:
    ts_triptych_v3(combined_df_daily_val, wet_start=wettest_start_val , wet_end=wettest_end_val, dry_start=dryest_start_val,dry_end=dryest_end_val, save_path="warm_springs_mts_daily_TS_validation.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d‑%b‑%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title = "Warm Springs MTS Daily Validation Timeseries Across Models", main_title_font=14, main_title_y = 0.99, main_title_pad = 0.05, row_titles=("Full Validation Period", "Most‑wet water‑year", "Most‑dry water‑year"))

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_validation:
    ts_triptych_v3((noPhys_val_csv_1H, phys_val_csv_1H, path_to_physics_data_1H), wet_start=wettest_start_val, wet_end=wettest_end_val, dry_start=dryest_start_val, dry_end=dryest_end_val, save_path="warm_springs_mts_hourly_TS_validation.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d-%b-%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title="Warm Springs MTS Hourly Validation Period Timeseries Across Models", main_title_font=14, main_title_y=0.99, main_title_pad=0.05, row_titles=("Full Validation period","Most-wet water-year","Most-dry water-year")) 

else:
    ts_triptych_v3(combined_df_hourly_val, wet_start=wettest_start_val, wet_end=wettest_end_val, dry_start=dryest_start_val, dry_end=dryest_end_val, save_path="warm_springs_mts_hourly_TS_validation.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d‑%b‑%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title = "Warm Springs MTS Hourly Validation Timeseries Across Models", main_title_font=14, main_title_y = 0.99, main_title_pad = 0.05, row_titles=("Full Validation Period", "Most‑wet water‑year", "Most‑dry water‑year"))

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_test:
    ts_triptych_v3((no_physics_test_csv_1D, physics_test_csv_1D, path_to_physics_data_1D), wet_start=wettest_start_test, wet_end=wettest_end_test, dry_start=dryest_start_test, dry_end=dryest_end_test, save_path="warm_springs_mts_daily_TS_test.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d-%b-%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title="Warm Springs MTS Daily Test Period Timeseries Across Models", main_title_font=14, main_title_y=0.99, main_title_pad=0.05, row_titles=("Full Test period","Most-wet water-year","Most-dry water-year"))
    
else:
    ts_triptych_v3(combined_df_daily, wet_start=wettest_start_test, wet_end=wettest_end_test, dry_start=dryest_start_test, dry_end=dryest_end_test, save_path="warm_springs_mts_daily_TS_test.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d‑%b‑%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title = "Warm Springs MTS Daily Test Timeseries Across Models", main_title_font=14, main_title_y = 0.99, main_title_pad = 0.05, row_titles=("Full Test Period", "Most‑wet water‑year", "Most‑dry water‑year"))

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_test:
    ts_triptych_v3((no_physics_test_csv_1H, physics_test_csv_1H, path_to_physics_data_1H), wet_start=wettest_start_test, wet_end=wettest_end_test, dry_start=dryest_start_test, dry_end=dryest_end_test, save_path="warm_springs_mts_hourly_TS_test.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d-%b-%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title="Warm Springs MTS Hourly Test Period Timeseries Across Models", main_title_font=14, main_title_y=0.99, main_title_pad=0.05, row_titles=("Full Test period","Most-wet water-year","Most-dry water-year"))
    
else:
    ts_triptych_v3(combined_df_hourly, wet_start=wettest_start_test, wet_end=wettest_end_test, dry_start=dryest_start_test, dry_end=dryest_end_test, save_path="warm_springs_mts_hourly_TS_test.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d‑%b‑%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title = "Warm Springs MTS Hourly Test Timeseries Across Models", main_title_font=14, main_title_y = 0.99, main_title_pad = 0.05, row_titles=("Full Test Period", "Most‑wet water‑year","Most‑dry water‑year"))

##### Gridded Scatter Plots - Test

In [ ]:
switch_ctx(RES_1D)

if not skip_best_model_test:
    scatter_pngs = scatter_triptych_pngs_v3((no_physics_test_csv_1D, physics_test_csv_1D, path_to_physics_data_1D), wet_start = wettest_start_test, wet_end = wettest_end_test, dry_start = dryest_start_test, dry_end = dryest_end_test, out_dir = "warm_springs_mts_daily_scatter", layout = "horizontal", square_side = 4.5, legend_font  = 16, axis_font = 16, point_size = 28, top_pad = .90, suptitle_y = 1.04, dpi = 600, row_titles = ("Warm Springs MTS Daily – Full test period", "Warm Springs MTS Daily – Wettest water‑year", "Warm Springs MTS Daily – Driest water‑year"), resolution="mts_daily")

else:
    scatter_pngs = scatter_triptych_pngs_v3(combined_df_daily, wet_start = wettest_start_test, wet_end = wettest_end_test, dry_start = dryest_start_test, dry_end = dryest_end_test, out_dir = "warm_springs_mts_daily_scatter", layout = "horizontal", square_side = 4.5,legend_font  = 16, axis_font = 16, point_size = 28, top_pad = .90, suptitle_y = 1.04, dpi = 600, row_titles = ("Warm Springs MTS Daily – Full test period", "Warm Springs MTS Daily – Wettest water‑year", "Warm Springs MTS Daily – Driest water‑year"), resolution="mts_daily")

In [ ]:
switch_ctx(RES_1H)

if not skip_best_model_test:
    scatter_pngs = scatter_triptych_pngs_v3((no_physics_test_csv_1H, physics_test_csv_1H, path_to_physics_data_1H), wet_start = wettest_start_test, wet_end = wettest_end_test, dry_start = dryest_start_test, dry_end = dryest_end_test, out_dir = "warm_springs_mts_hourly_scatter", layout = "horizontal", square_side = 4.5, legend_font = 16, axis_font = 16, point_size = 28, top_pad = .90, suptitle_y = 1.04, dpi = 600, row_titles = ("Warm Springs MTS Hourly – Full test period", "Warm Springs MTS Hourly – Wettest water‑year", "Warm Springs MTS Hourly – Driest water‑year"), resolution="mts_hourly")
    
else:
    scatter_pngs = scatter_triptych_pngs_v3(combined_df_hourly, wet_start = wettest_start_test, wet_end = wettest_end_test, dry_start = dryest_start_test,  dry_end = dryest_end_test, out_dir = "warm_springs_mts_hourly_scatter", layout = "horizontal", square_side = 4.5, legend_font = 16, axis_font = 16, point_size = 28, top_pad = .90, suptitle_y = 1.04, dpi = 600, row_titles = ("Warm Springs MTS Hourly – Full test period", "Warm Springs MTS Hourly – Wettest water‑year", "Warm Springs MTS Hourly – Driest water‑year"), resolution="mts_hourly")